# الفصل 55: تحسين الاستدلال المتقدم

> معرفة KV Cache وحدها لا تكفي لتحقيق أداء إنتاجي تنافسي.

- **الجزء:** 11 — النشر والتشغيل
- **الفصل:** 55
- **المستوى:** خبير
- **زمن القراءة:** 15 دقيقة
- **المتطلبات:** الفصل 54: أساسيات تقديم النماذج للاستدلال
- **الوسوم:** Quantization، Speculative Decoding، Continuous Batching، TensorRT-LLM

## كيف تشغّل هذا الدفتر؟

1. **Google Colab:** ارفع الملف عبر `File → Upload notebook`، ثم `Runtime → Run all`.
2. **Kaggle:** أنشئ Notebook ثم `File → Import notebook`، ثم `Run All`.
3. شغّل خلية **إعداد البيئة** أولًا؛ ستفحص الحزم وتثبّت الناقص تلقائيًا.
4. للتدريب أو النماذج الكبيرة: Colab `Runtime → Change runtime type → T4 GPU`، وKaggle `Settings → Accelerator → GPU`.


<div dir="rtl" style="margin:2px 0 18px;line-height:2.4;">
<b>افتح هذا الدفتر على منصة مجانية:</b><br>
<a href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-11-llmops/ch55-inference-optimization.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#F9AB00;color:#111111;">Google Colab</a> <a href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-11-llmops%2Fch55-inference-optimization.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#20BEFF;color:#FFFFFF;">Kaggle</a> <a href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-11-llmops/ch55-inference-optimization.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#579ACA;color:#FFFFFF;">Binder</a> <a href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-11-llmops/ch55-inference-optimization.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#6C5CE7;color:#FFFFFF;">Gradient</a> <a href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-11-llmops/ch55-inference-optimization.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#37474F;color:#FFFFFF;">nbviewer</a> <a href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-11-llmops/ch55-inference-optimization.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#24292F;color:#FFFFFF;">GitHub</a>
</div>


## ملاحظات التشغيل

- **الحزم:** لا توجد مكتبات خارجية مطلوبة.
- **مقاطع توضيحية:** بعض الشيفرة تحتوي `...` أو `NotImplementedError`؛ هذا مقصود لبيان البنية وليس خطأً في الدفتر.


<style>
/* نص عربي من اليمين إلى اليسار، مع إبقاء الشيفرة من اليسار */
.rendered_html, .jp-RenderedHTMLCommon, .text_cell_render, .markdown-body {
  direction: rtl;
  text-align: right;
}
.rendered_html pre, .rendered_html code,
.jp-RenderedHTMLCommon pre, .jp-RenderedHTMLCommon code,
.text_cell_render pre, .text_cell_render code,
.markdown-body pre, .markdown-body code {
  direction: ltr;
  text-align: left;
  unicode-bidi: embed;
}
.rendered_html table, .jp-RenderedHTMLCommon table, .text_cell_render table {
  direction: rtl;
}
</style>


In [ ]:
# === إعداد البيئة ===
# يفحص هذا القسم الحزم المطلوبة ويحاول تثبيت الناقص منها تلقائيًا.
# شغّل هذه الخلية أولًا، ثم بقية خلايا الدفتر.
import importlib.util
import subprocess
import sys

print("Python:", sys.version.split()[0])

# (اسم وحدة الاستيراد، اسم حزمة pip)
REQUIRED = [

]
AUTO_INSTALL = {"PIL", "bs4", "datasets", "faiss", "fasttext", "librosa", "networkx", "opentelemetry", "peft", "prometheus_client", "pydantic", "rank_bm25", "requests", "sentence_transformers", "sentencepiece", "soundfile", "tokenizers", "trafilatura", "transformers", "wandb", "yaml"}


def has_module(name):
    try:
        return importlib.util.find_spec(name) is not None
    except Exception:
        return False


if not REQUIRED:
    print("لا توجد مكتبات خارجية مطلوبة؛ يكفي Python القياسي.")
else:
    missing = [(module, package) for module, package in REQUIRED if not has_module(module)]
    auto_specs = []
    manual_specs = []
    for module, package in missing:
        specs = package.split()
        if module in AUTO_INSTALL:
            auto_specs.extend(specs)
        else:
            manual_specs.extend(specs)
    auto_specs = list(dict.fromkeys(auto_specs))
    manual_specs = list(dict.fromkeys(manual_specs))

    if auto_specs:
        print("⏳ تثبيت تلقائي:", " ".join(auto_specs))
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *auto_specs])
            print("✅ تم التثبيت.")
        except Exception as exc:
            print("⚠️ تعذّر التثبيت التلقائي:", exc)
            print("شغّل يدويًا: %pip install", " ".join(auto_specs))
    if manual_specs:
        print("⚠️ حزم تحتاج تثبيتًا يدويًا:", " ".join(manual_specs))
        print("شغّل: %pip install", " ".join(manual_specs))
    if not auto_specs and not manual_specs:
        print("✅ كل الحزم المطلوبة متاحة.")

    print("\nالحالة النهائية:")
    for module, package in REQUIRED:
        print(("✅" if has_module(module) else "⚠️"), package)


<div class="notebook-actions" dir="rtl"><span class="notebook-actions__label">شغّل الدفتر:</span><a class="nb-action nb-action--colab" href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-11-llmops/ch55-inference-optimization.ipynb" target="_blank" rel="noopener" title="افتح في Google Colab — GPU مجاني">Google Colab</a><a class="nb-action nb-action--kaggle" href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-11-llmops%2Fch55-inference-optimization.ipynb" target="_blank" rel="noopener" title="افتح في Kaggle — GPU/TPU مجاني">Kaggle</a><a class="nb-action nb-action--binder" href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-11-llmops/ch55-inference-optimization.ipynb" target="_blank" rel="noopener" title="افتح في Binder — بيئة Jupyter مجانية">Binder</a><a class="nb-action nb-action--gradient" href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-11-llmops/ch55-inference-optimization.ipynb" target="_blank" rel="noopener" title="افتح في Paperspace Gradient — GPU مجاني محدود">Gradient</a><a class="nb-action nb-action--nbviewer" href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-11-llmops/ch55-inference-optimization.ipynb" target="_blank" rel="noopener" title="اعرض الدفتر دون تنفيذ">nbviewer</a><a class="nb-action nb-action--github" href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-11-llmops/ch55-inference-optimization.ipynb" target="_blank" rel="noopener" title="اعرض الملف المصدر على GitHub">GitHub</a></div>

> **ملخص الفصل**
>
> معرفة KV Cache وحدها لا تكفي لتحقيق أداء إنتاجي تنافسي.

> **بيانات الفصل**
>
> - **الجزء:** 11 — النشر والتشغيل
> - **المستوى:** خبير
> - **زمن القراءة:** 15 دقيقة
> - **المتطلبات:** الفصل 54: أساسيات تقديم النماذج للاستدلال


## مخرجات التعلم

بنهاية هذا الفصل ستكون قادرًا على:

- فهم PagedAttention — حل مشكلة تجزّؤ الذاكرة
- فهم Continuous Batching — حل مشكلة التجميع الساكن
- فهم Quantization


## 1. لماذا يهمّك هذا الموضوع؟

معرفة KV Cache وحدها لا تكفي لتحقيق أداء إنتاجي تنافسي. هذا الفصل يكشف "الأسرار الهندسية" وراء المحركات الحديثة التي تحقق throughput أعلى بعشرات المرات من التطبيق الساذج.


## 2. الخريطة الذهنية


```text
تحسين الاستدلال
├── كفاءة الذاكرة
│   ├── PagedAttention: صفحات KV ديناميكية
│   ├── Prefix Caching: مشاركة البادئات
│   └── GQA/MQA: تقليل رؤوس KV
├── كفاءة الجدولة
│   ├── Continuous Batching
│   ├── Chunked Prefill
│   └── أولويات الطلبات
├── كفاءة الحوسبة
│   ├── FlashAttention
│   ├── التكميم (FP8/INT8/INT4)
│   └── Speculative Decoding
└── التوازي
    ├── Tensor Parallelism
    ├── Pipeline Parallelism
    └── Replicas خلف موازن تحميل
```


## 3. الشرح المتدرج

### 🟢 أ) PagedAttention — حل مشكلة تجزّؤ الذاكرة

المشكلة: حجوزات KV Cache التقليدية تُخصِّص ذاكرة متجاورة (contiguous) بحجم أقصى سياق مُحتمَل لكل طلب، حتى لو كان الطلب الفعلي أقصر بكثير — هذا يُهدر ذاكرة GPU الثمينة بشدة (قد يصل الهدر لـ 60-80%).


```text
التخصيص التقليدي (مُهدر):
طلب 1 (طول فعلي: 50) -> [████████████████████ محجوز بالكامل حتى 2048]
طلب 2 (طول فعلي: 1800) -> [████████████████████ محجوز بالكامل حتى 2048]

PagedAttention (مستوحى من الـ Virtual Memory في أنظمة التشغيل):
طلب 1 -> [██░░░░░░░░] (صفحات حسب الحاجة الفعلية فقط)
طلب 2 -> [█████████░]
الذاكرة المُحرَّرة تُعاد استخدامها فورًا لطلبات أخرى
```


In [ ]:
class PagedKVCacheConceptual:
    """
    توضيح مفاهيمي لفكرة PagedAttention:
    بدلاً من حجز ذاكرة متجاورة ضخمة لكل تسلسل، نُقسّم KV Cache
    إلى "صفحات" (blocks) ثابتة الحجم، تُخصَّص ديناميكيًا عند الحاجة فقط
    (تمامًا مثل Paging في أنظمة التشغيل)
    """
    def __init__(self, block_size=16, total_blocks=1000, head_dim=128, n_heads=8):
        self.block_size = block_size
        self.physical_blocks = torch.zeros(total_blocks, block_size, n_heads, head_dim * 2)
        self.free_blocks = list(range(total_blocks))
        self.block_tables = {}   # {sequence_id: [block_ids]} — خريطة منطقي -> فعلي

    def allocate_for_sequence(self, seq_id: str, num_tokens: int):
        num_blocks_needed = (num_tokens + self.block_size - 1) // self.block_size
        allocated = [self.free_blocks.pop() for _ in range(num_blocks_needed)]
        self.block_tables[seq_id] = allocated
        return allocated

    def free_sequence(self, seq_id: str):
        """تحرير فوري للذاكرة بمجرد انتهاء التسلسل — يُعاد استخدامها لطلبات جديدة فورًا"""
        self.free_blocks.extend(self.block_tables.pop(seq_id))

    def share_blocks_for_prefix(self, seq_id_a: str, seq_id_b: str, shared_prefix_len: int):
        """
        ميزة إضافية قوية: مشاركة الصفحات الفعلية لنفس البادئة
        (مثل system prompt مشترك) بين عدة طلبات دون نسخ! (Copy-on-Write)
        """
        shared_blocks_count = shared_prefix_len // self.block_size
        self.block_tables[seq_id_b][:shared_blocks_count] = \
            self.block_tables[seq_id_a][:shared_blocks_count]


> 💡 **الفائدة العملية**: PagedAttention يسمح بزيادة حجم الدفعة الفعّالة (effective batch size) بشكل كبير جدًا بنفس كمية ذاكرة GPU، لأن الهدر شبه معدوم — هذا هو السبب الرئيسي وراء تفوّق throughput في vLLM.

### 🟡 ب) Continuous Batching — حل مشكلة التجميع الساكن

في التجميع التقليدي (Static Batching)، ننتظر اكتمال **كل** الطلبات في الدفعة قبل بدء دفعة جديدة — إذا انتهى طلب قصير بينما طلب آخر طويل لا يزال يُولّد، تبقى GPU "منتظِرة" الطلب الأطول بلا فائدة.


In [ ]:
class ContinuousBatchingScheduler:
    """
    توضيح مفاهيمي: بمجرد انتهاء أي تسلسل في الدفعة،
    يُستبدل فورًا بطلب جديد من قائمة الانتظار، دون انتظار بقية الدفعة
    """
    def __init__(self, max_batch_size=32):
        self.max_batch_size = max_batch_size
        self.active_sequences = {}   # {seq_id: state}
        self.waiting_queue = []

    def step(self, model):
        """خطوة واحدة من التوليد عبر كل التسلسلات النشطة معًا"""
        # ملء الدفعة من قائمة الانتظار إن وُجدت مساحة فارغة
        while len(self.active_sequences) < self.max_batch_size and self.waiting_queue:
            new_seq = self.waiting_queue.pop(0)
            self.active_sequences[new_seq["id"]] = new_seq

        if not self.active_sequences:
            return

        # تمريرة أمامية واحدة لكل التسلسلات النشطة دفعة واحدة (batched forward pass)
        batch_input = self._prepare_batch(self.active_sequences)
        next_tokens = model.forward_step(batch_input)   # token واحد جديد لكل تسلسل

        # تحديث الحالة وإزالة أي تسلسل اكتمل فورًا (تحرير مكانه للتالي في الطابور)
        for seq_id, token in next_tokens.items():
            self.active_sequences[seq_id]["tokens"].append(token)
            if token == EOS_TOKEN_ID or len(self.active_sequences[seq_id]["tokens"]) >= \
               self.active_sequences[seq_id]["max_tokens"]:
                completed = self.active_sequences.pop(seq_id)
                self._return_result(completed)

    def _prepare_batch(self, sequences): ...  # تفاصيل التجميع
    def _return_result(self, completed_seq): ...  # إرجاع النتيجة للمستخدم


### 🔴 ج) طبقة التعمّق: التشفير التخميني (Speculative Decoding)

تقنية ذكية جدًا لتسريع الاستدلال دون المساس بجودة المخرجات إطلاقًا: استخدام نموذج **صغير وسريع** (Draft Model) لاقتراح عدة tokens مستقبلية دفعة واحدة، ثم التحقق منها بالنموذج **الكبير الدقيق** بتمريرة واحدة فقط.


In [ ]:
@torch.no_grad()
def speculative_decoding_step(draft_model, target_model, tokenizer, context_ids,
                                num_speculative_tokens=4):
    """
    1. النموذج الصغير (draft) يُولّد K رموز مقترحة بسرعة (تسلسليًا لكنه سريع جدًا)
    2. النموذج الكبير (target) يتحقق من كل الرموز K دفعة واحدة (تمريرة أمامية واحدة فقط!)
    3. نقبل الرموز المتطابقة مع توزيع target، ونرفض ونُصحّح من أول اختلاف
    """
    # الخطوة 1: اقتراح سريع من النموذج الصغير
    draft_tokens = []
    draft_probs = []
    current_ids = context_ids.clone()

    for _ in range(num_speculative_tokens):
        logits = draft_model(current_ids).logits[:, -1, :]
        probs = F.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        draft_tokens.append(next_token)
        draft_probs.append(probs.gather(-1, next_token))
        current_ids = torch.cat([current_ids, next_token], dim=1)

    # الخطوة 2: تحقق دفعة واحدة من النموذج الكبير (تمريرة أمامية واحدة لكل الرموز المقترحة)
    full_sequence = current_ids
    target_logits = target_model(full_sequence).logits
    target_probs_all = F.softmax(target_logits, dim=-1)

    # الخطوة 3: قبول/رفض كل رمز مقترح (خوارزمية Rejection Sampling)
    accepted_tokens = []
    context_len = context_ids.shape[1]

    for i, (draft_token, draft_prob) in enumerate(zip(draft_tokens, draft_probs)):
        position = context_len + i - 1
        target_prob = target_probs_all[:, position, :].gather(-1, draft_token)

        acceptance_ratio = torch.min(torch.ones_like(target_prob), target_prob / draft_prob)
        if torch.rand_like(acceptance_ratio) < acceptance_ratio:
            accepted_tokens.append(draft_token)
        else:
            # رفض: نستبدل برمز جديد من توزيع "الفرق" المُصحَّح إحصائيًا
            corrected_dist = torch.clamp(
                target_probs_all[:, position, :] - draft_probs[i], min=0
            )
            corrected_dist = corrected_dist / corrected_dist.sum()
            correction_token = torch.multinomial(corrected_dist, num_samples=1)
            accepted_tokens.append(correction_token)
            break   # نتوقف عند أول رفض

    return torch.cat(accepted_tokens, dim=1) if accepted_tokens else draft_tokens[0]


🟢 **الحدس الرياضي الجميل**: خوارزمية القبول/الرفض مصمَّمة رياضيًا لضمان أن **التوزيع النهائي للرموز المقبولة مطابق تمامًا** لتوزيع النموذج الكبير وحده — أي أن Speculative Decoding **لا يُضحّي بأي جودة إطلاقًا**، فقط يُسرِّع العملية الحسابية. هذا ممكن لأن التحقق (forward pass واحد لعدة رموز) أرخص بكثير من توليدها تسلسليًا واحدًا تلو الآخر.

> 💡 **متى يكون Speculative Decoding فعّالًا؟** عندما يكون النموذج الصغير (draft) **متوافقًا جيدًا** (well-aligned) مع توزيع النموذج الكبير (أفضل حالة: نفس العائلة، نموذج مُقطَّر منه). كلما زاد معدل القبول (acceptance rate)، زادت المكاسب — في أفضل الحالات تصل التسريعات لـ 2-3x بدون أي خسارة جودة.


## 4. مثال محلول خطوة بخطوة

**مثال Speculative Decoding:** النموذج الصغير يقترح 4 رموز، والنموذج الكبير يتحقق منها بتمريرة واحدة.

لنفترض احتمالات القبول: \(p_1=0.9\)، \(p_2=0.8\)، \(p_3=0.7\)، \(p_4=0.6\).

**عدد الرموز المقبولة المتوقع:**

\[
E = 1 + p_1 + p_1p_2 + p_1p_2p_3 + p_1p_2p_3p_4
\]

\[
E = 1 + 0.9 + 0.72 + 0.504 + 0.3024 = 3.4264
\]

أي نحو 3.43 token لكل تمريرة للنموذج الكبير، بدلًا من token واحد.

**التسريع:** إذا كانت تكلفة النموذج الصغير 15% من تكلفة الكبير لكل token، فإن:

\[
\text{Speedup} \approx \frac{3.4264}{1 + 4 \times 0.15} = \frac{3.4264}{1.6} \approx 2.14\times
\]

| معدل القبول | رموز متوقعة | تسريع تقريبي |
|---|---:|---:|
| منخفض (0.5) | 1.94 | 1.2× |
| متوسط (0.75) | 2.73 | 1.7× |
| مرتفع (0.9) | 3.43 | 2.1× |

**الدرس:** الفائدة تعتمد على توافق النموذجين. إذا كان النموذج الصغير ضعيفًا، يرفض الكبير معظم الاقتراحات، وقد يصبح speculative decoding أبطأ من التوليد العادي بسبب تكلفة الاقتراح.


## 5. نصائح وحيل

> 💡 **اضبط حجم الصفحة (Block Size)**: 16 token شائع، لكن جرّب 8/32 حسب طول الطلبات. الصفحات الأصغر تقلّل الهدر الداخلي، والأكبر تقلّل overhead الجدولة.

> 💡 **جدول بالـ tokens لا بالطلبات**: Continuous batching يجب أن يوازن بين عدد الطلبات وعدد tokens في الدفعة. دفعة كبيرة جدًا تزيد زمن أول token.

> 💡 **استخدم Chunked Prefill للسياقات الطويلة**: تقسيم الـ prompt الطويل إلى أجزاء يمنع تجميد الطلبات القصيرة خلفه.

> 💡 **اختر draft model من نفس العائلة**: التوافق في tokenizer والتوزيع يرفع معدل القبول. لا تستخدم نموذجًا مختلف المعجم.

> 💡 **قِس معدل القبول قبل التفعيل**: إذا كان أقل من 60%، فغالبًا لن تستفيد. راقب `acceptance_rate` و`accepted_tokens_per_step`.


## 6. أخطاء شائعة ومصائد

> ⚠️ **تفعيل Speculative Decoding مع tokenizer مختلف**: الاقتراحات قد لا تتطابق مع رموز النموذج الكبير، فينهار معدل القبول.

> ⚠️ **الإفراط في عدد الرموز المقترحة**: 8 رموز قد ترفع التكلفة دون مكسب إذا كان القبول منخفضًا. ابدأ بـ 3-5.

> ⚠️ **إهمال Prefill**: معظم التحسينات تركز على Decode، لكن الطلبات الطويلة تقضي وقتًا كبيرًا في Prefill. استخدم chunked prefill وprefix caching.

> ⚠️ **الاعتقاد أن PagedAttention يلغي الحاجة لتخطيط السعة**: يقلّل الهدر، لكنه لا يخلق ذاكرة. ما زلت تحتاج حساب KV Cache.

> ⚠️ **قياس throughput دون latency**: قد ترفع الدفعة throughput الكلي لكن تُدهور p99. وازن بينهما حسب نوع التطبيق.


## 7. جدول مقارنة

| التقنية | المبدأ | التسريع النموذجي | أثر على الجودة |
|---|---|---|---|
| **KV Cache** | تجنّب إعادة الحساب | أساسي (ضروري جدًا) | معدوم |
| **PagedAttention** | إدارة ذاكرة فعّالة شبيهة بـ Virtual Memory | +throughput كبير (عبر batch أكبر) | معدوم |
| **Continuous Batching** | استبدال ديناميكي للتسلسلات المكتملة | +throughput كبير | معدوم |
| **Speculative Decoding** | تحقق دفعي بدلاً من توليد تسلسلي | 2-3x | معدوم (رياضيًا مضمون) |
| **Quantization (الاستدلال)** | تقليل دقة الأوزان/التنشيط | 1.5-4x + ذاكرة أقل | طفيف (حسب الطريقة) |
| **Flash Attention** (الفصل 14) | تقليل قراءة/كتابة الذاكرة | أساسي للسياقات الطويلة | معدوم |


## 8. ملخص أفضل الممارسات

- [ ] تأكد من استخدام محرك يدعم PagedAttention وContinuous Batching (vLLM/TGI)
- [ ] قيّم Speculative Decoding إذا توفّر نموذج صغير متوافق من نفس العائلة
- [ ] راقب معدل القبول (acceptance rate) عند استخدام Speculative Decoding لضمان فعاليته
- [ ] اختبر Prefix Caching لحالات الاستخدام ذات System Prompts الثابتة الطويلة

---


## 9. أسئلة مقابلات

**سؤال 1: كيف يحل PagedAttention مشكلة تجزئة الذاكرة؟**

> **إجابة نموذجية:** التخصيص التقليدي يحجز لكل تسلسل مساحة متجاورة بحجم أقصى سياق محتمل، فيُهدر مساحة كبيرة إذا كان التسلسل أقصر. PagedAttention يقسّم KV Cache إلى صفحات ثابتة الحجم، ويخصّصها ديناميكيًا عند الحاجة، مع جدول يربط الصفحات المنطقية بالفيزيائية. هذا يقلّل الهدر الداخلي ويسمح بمشاركة الصفحات بين الطلبات (Copy-on-Write) للبادئات المشتركة، فيزيد الدفعة الفعّالة.

**سؤال 2: ما الفرق بين Static وContinuous Batching؟**

> **إجابة نموذجية:** Static batching ينتظر اكتمال كل طلبات الدفعة قبل بدء دفعة جديدة؛ إذا انتهى طلب قصير، تبقى GPU تنتظر الأطول. Continuous batching يُدخل طلبًا جديدًا فور انتهاء أي طلب، على مستوى كل خطوة توليد. النتيجة: استخدام أعلى للـ GPU وزمن انتظار أقل، خاصة مع تفاوت أطوال الطلبات.

**سؤال 3: هل Speculative Decoding يغيّر جودة المخرجات؟**

> **إجابة نموذجية:** لا، إذا طُبّق بشكل صحيح. خوارزمية القبول/الرفض مصممة بحيث يكون التوزيع النهائي للرموز مطابقًا لتوزيع النموذج الكبير. النموذج الصغير يقترح فقط، والنموذج الكبير يقرر. أي تطبيق خاطئ (مثل قبول كل الاقتراحات دون تصحيح) يغيّر التوزيع ويُضعف الجودة.


## 10. تمرين عملي

**الهدف:** محاكاة أثر Continuous Batching على استخدام GPU.

**الخطوات:**
1. أنشئ 100 طلب بأطوال خرج عشوائية بين 20 و500 token.
2. افترض دفعة قصوى 8 طلبات، وزمن خطوة ثابتًا (مثل 20ms).
3. احسب زمن الإنجاز الكلي في وضع Static Batching (انتظار اكتمال الدفعة).
4. احسبه في وضع Continuous Batching (إدخال طلب جديد فور انتهاء واحد).
5. احسب نسبة الوقت المهدر (GPU idle) في كل وضع.
6. ارسم النتائج في جدول، واكتب استنتاجًا.

**معيار النجاح:** إظهار تحسّن واضح في الزمن الكلي مع Continuous Batching، وتفسير العلاقة بين تفاوت الأطوال ونسبة الهدر.


## 11. مراجع للتعمّق

- Kwon, W., et al. (2023). *PagedAttention and vLLM*. SOSP.
- Yu, G.-I., et al. (2022). *Orca: A Distributed Serving System for Transformer-Based Generative Models*. OSDI.
- Leviathan, Y., Kalman, M., & Matias, Y. (2023). *Fast Inference from Transformers via Speculative Decoding*. ICML.
- Chen, C., et al. (2023). *Accelerating Large Language Model Decoding with Speculative Sampling*. arXiv:2302.01318.
- Dao, T. (2023). *FlashAttention-2: Faster Attention with Better Parallelism and Work Partitioning*. ICLR 2024.


## 🧪 مساحة تجربة

استخدم الخلية التالية لتجربة أفكار الفصل أو حلّ التمرين. عدّل القيم وأعد تشغيل الخلية.


In [ ]:
# 🧪 مساحة تجربة
# اكتب هنا حلّ التمرين أو أي تجربة إضافية.


---
📘 **المصدر:** كتاب *إتقان المحوّلات والنماذج اللغوية الكبيرة* — النشر والتشغيل / الفصل 55: تحسين الاستدلال المتقدم.

هذا الدفتر مولّد آليًا من ملف الفصل؛ لا تعدّله يدويًا، بل عدّل المصدر ثم أعد البناء.
